# 04 — Feature Engineering for Uplift Modeling

## Objective

This notebook transforms the cleaned Criteo Uplift dataset into a machine-learning-ready feature set.

The objectives are:

- Identify predictive customer features
- Separate treatment from predictive features
- Analyze feature distributions
- Create robust numerical transformations
- Create interaction features
- Prepare features for machine learning
- Prevent treatment leakage
- Prepare separate datasets for predictive and uplift modeling

A key principle of uplift modeling is that treatment assignment must be handled separately from customer covariates. Treatment will therefore not be included as an ordinary predictive feature in the base feature matrix.

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
DATA_PATH = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\data\raw\criteo-research-uplift-v2.1.csv.gz"
)

print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print(
        "Dataset size (MB):",
        round(DATA_PATH.stat().st_size / (1024 ** 2), 2)
    )

Dataset exists: True
Dataset size (MB): 297.0


In [5]:
SAMPLE_SIZE = 100_000

df = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    nrows=SAMPLE_SIZE
)

print("Sample shape:", df.shape)
df.head()

Sample shape: (100000, 16)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [6]:
feature_cols = [f"f{i}" for i in range(12)]

treatment_col = "treatment"

outcome_cols = [
    "conversion",
    "visit",
    "exposure"
]

print("Features:", feature_cols)
print("Treatment:", treatment_col)
print("Outcomes:", outcome_cols)

Features: ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']
Treatment: treatment
Outcomes: ['conversion', 'visit', 'exposure']


In [7]:
X_raw = df[feature_cols].copy()

treatment = df[treatment_col].copy()

y_conversion = df["conversion"].copy()
y_visit = df["visit"].copy()
y_exposure = df["exposure"].copy()

print("X shape:", X_raw.shape)
print("Treatment shape:", treatment.shape)
print("Conversion shape:", y_conversion.shape)

X shape: (100000, 12)
Treatment shape: (100000,)
Conversion shape: (100000,)


In [8]:
X_raw.dtypes

f0     float64
f1     float64
f2     float64
f3     float64
f4     float64
f5     float64
f6     float64
f7     float64
f8     float64
f9     float64
f10    float64
f11    float64
dtype: object

In [9]:
missing_features = X_raw.isnull().sum()

print("Missing values:")
print(missing_features)

print(
    "\nTotal missing values:",
    missing_features.sum()
)

Missing values:
f0     0
f1     0
f2     0
f3     0
f4     0
f5     0
f6     0
f7     0
f8     0
f9     0
f10    0
f11    0
dtype: int64

Total missing values: 0


In [10]:
infinite_counts = np.isinf(X_raw.select_dtypes(include=np.number)).sum()

print(infinite_counts)

print(
    "\nTotal infinite values:",
    infinite_counts.sum()
)

f0     0
f1     0
f2     0
f3     0
f4     0
f5     0
f6     0
f7     0
f8     0
f9     0
f10    0
f11    0
dtype: int64

Total infinite values: 0


In [11]:
feature_stats = X_raw.describe().T

feature_stats

,count,mean,std,min,25%,50%,75%,max
f0,100000.0,21.598848,4.604217,12.616365,20.599416,22.959279,24.667063,26.745092
f1,100000.0,10.067764,0.099210,10.059654,10.059654,10.059654,10.059654,14.560990
f2,100000.0,8.346235,0.256821,8.214383,8.214383,8.214383,8.231230,9.051956
f3,100000.0,4.348703,1.131443,-6.314955,4.679882,4.679882,4.679882,4.679882
f4,100000.0,10.363419,0.463894,10.280525,10.280525,10.280525,10.280525,20.036060
f5,100000.0,4.034308,0.421120,-6.729900,4.115453,4.115453,4.115453,4.115453
f6,100000.0,-4.478084,4.147827,-25.860481,-6.699321,-3.282109,-1.288207,0.294443
f7,100000.0,5.091506,1.186038,4.833815,4.833815,4.833815,4.833815,11.995830
f8,100000.0,3.952916,0.042551,3.640858,3.955396,3.971858,3.971858,3.971858
f9,100000.0,14.419157,4.676643,13.190056,13.190056,13.190056,13.190056,67.419043


In [12]:
scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X_raw),
    columns=feature_cols,
    index=X_raw.index
)

X_scaled.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11
0,-1.950935,-0.081747,2.453839,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709
1,-1.950935,-0.081747,2.556092,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709
2,-1.950935,-0.081747,2.408462,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709
3,-1.950935,-0.081747,2.556526,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709
4,-1.950935,-0.081747,2.693578,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709


In [13]:
print("Original means:")
print(X_raw.mean())

print("\nScaled means:")
print(X_scaled.mean().round(6))

Original means:
f0     21.598848
f1     10.067764
f2      8.346235
f3      4.348703
f4     10.363419
f5      4.034308
f6     -4.478084
f7      5.091506
f8      3.952916
f9     14.419157
f10     5.334712
f11    -0.172133
dtype: float64

Scaled means:
f0    -0.0
f1    -0.0
f2     0.0
f3    -0.0
f4    -0.0
f5    -0.0
f6    -0.0
f7    -0.0
f8     0.0
f9     0.0
f10   -0.0
f11   -0.0
dtype: float64


In [14]:
print("Original standard deviations:")
print(X_raw.std())

print("\nScaled standard deviations:")
print(X_scaled.std().round(6))

Original standard deviations:
f0     4.604217
f1     0.099210
f2     0.256821
f3     1.131443
f4     0.463894
f5     0.421120
f6     4.147827
f7     1.186038
f8     0.042551
f9     4.676643
f10    0.170604
f11    0.031479
dtype: float64

Scaled standard deviations:
f0     1.000005
f1     1.000005
f2     1.000005
f3     1.000005
f4     1.000005
f5     1.000005
f6     1.000005
f7     1.000005
f8     1.000005
f9     1.000005
f10    1.000005
f11    1.000005
dtype: float64


In [15]:
X_rank = X_raw.rank(pct=True)

X_rank.columns = [
    f"{col}_rank"
    for col in X_rank.columns
]

X_rank.head()

,f0_rank,f1_rank,f2_rank,f3_rank,f4_rank,f5_rank,f6_rank,f7_rank,f8_rank,f9_rank,f10_rank,f11_rank
0,0.06492,0.495405,0.95371,0.55934,0.47659,0.5249,0.93509,0.47511,0.2328,0.451065,0.47659,0.50853
1,0.06492,0.495405,0.96766,0.55934,0.47659,0.5249,0.93509,0.47511,0.2328,0.451065,0.47659,0.50853
2,0.06492,0.495405,0.94760,0.55934,0.47659,0.5249,0.93509,0.47511,0.2328,0.451065,0.47659,0.50853
3,0.06492,0.495405,0.96786,0.55934,0.47659,0.5249,0.93509,0.47511,0.2328,0.451065,0.47659,0.50853
4,0.06492,0.495405,0.99079,0.55934,0.47659,0.5249,0.93509,0.47511,0.2328,0.451065,0.47659,0.50853


In [16]:
interaction_data = X_scaled.copy()

interaction_data["f0_f1_interaction"] = (
    interaction_data["f0"] *
    interaction_data["f1"]
)

interaction_data["f2_f3_interaction"] = (
    interaction_data["f2"] *
    interaction_data["f3"]
)

interaction_data["f4_f5_interaction"] = (
    interaction_data["f4"] *
    interaction_data["f5"]
)

interaction_data["f6_f7_interaction"] = (
    interaction_data["f6"] *
    interaction_data["f7"]
)

interaction_data["f8_f9_interaction"] = (
    interaction_data["f8"] *
    interaction_data["f9"]
)

interaction_data["f10_f11_interaction"] = (
    interaction_data["f10"] *
    interaction_data["f11"]
)

interaction_data.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f0_f1_interaction,f2_f3_interaction,f4_f5_interaction,f6_f7_interaction,f8_f9_interaction,f10_f11_interaction
0,-1.950935,-0.081747,2.453839,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.159482,0.718253,-0.034433,-0.249996,-0.015319,-0.022081
1,-1.950935,-0.081747,2.556092,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.159482,0.748183,-0.034433,-0.249996,-0.015319,-0.022081
2,-1.950935,-0.081747,2.408462,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.159482,0.704971,-0.034433,-0.249996,-0.015319,-0.022081
3,-1.950935,-0.081747,2.556526,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.159482,0.748310,-0.034433,-0.249996,-0.015319,-0.022081
4,-1.950935,-0.081747,2.693578,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.159482,0.788426,-0.034433,-0.249996,-0.015319,-0.022081


In [17]:
X_engineered = X_scaled.copy()

X_engineered["feature_mean"] = X_scaled.mean(axis=1)
X_engineered["feature_std"] = X_scaled.std(axis=1)
X_engineered["feature_min"] = X_scaled.min(axis=1)
X_engineered["feature_max"] = X_scaled.max(axis=1)
X_engineered["feature_median"] = X_scaled.median(axis=1)

X_engineered.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,feature_mean,feature_std,feature_min,feature_max,feature_median
0,-1.950935,-0.081747,2.453839,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.113759,1.014877,-1.950935,2.453839,-0.01173
1,-1.950935,-0.081747,2.556092,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.122280,1.036510,-1.950935,2.556092,-0.01173
2,-1.950935,-0.081747,2.408462,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.109978,1.005406,-1.950935,2.408462,-0.01173
3,-1.950935,-0.081747,2.556526,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.122317,1.036603,-1.950935,2.556526,-0.01173
4,-1.950935,-0.081747,2.693578,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,-0.201269,0.109709,0.133738,1.066193,-1.950935,2.693578,-0.01173


In [18]:
X_final = pd.concat(
    [
        X_scaled,
        X_rank,
        X_engineered[
            [
                "feature_mean",
                "feature_std",
                "feature_min",
                "feature_max",
                "feature_median"
            ]
        ],
        interaction_data[
            [
                "f0_f1_interaction",
                "f2_f3_interaction",
                "f4_f5_interaction",
                "f6_f7_interaction",
                "f8_f9_interaction",
                "f10_f11_interaction"
            ]
        ]
    ],
    axis=1
)

print("Final feature matrix shape:", X_final.shape)
X_final.head()

Final feature matrix shape: (100000, 35)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,feature_std,feature_min,feature_max,feature_median,f0_f1_interaction,f2_f3_interaction,f4_f5_interaction,f6_f7_interaction,f8_f9_interaction,f10_f11_interaction
0,-1.950935,-0.081747,2.453839,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,1.014877,-1.950935,2.453839,-0.01173,0.159482,0.718253,-0.034433,-0.249996,-0.015319,-0.022081
1,-1.950935,-0.081747,2.556092,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,1.036510,-1.950935,2.556092,-0.01173,0.159482,0.748183,-0.034433,-0.249996,-0.015319,-0.022081
2,-1.950935,-0.081747,2.408462,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,1.005406,-1.950935,2.408462,-0.01173,0.159482,0.704971,-0.034433,-0.249996,-0.015319,-0.022081
3,-1.950935,-0.081747,2.556526,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,1.036603,-1.950935,2.556526,-0.01173,0.159482,0.748310,-0.034433,-0.249996,-0.015319,-0.022081
4,-1.950935,-0.081747,2.693578,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,1.066193,-1.950935,2.693578,-0.01173,0.159482,0.788426,-0.034433,-0.249996,-0.015319,-0.022081


In [19]:
print("Treatment column present:", "treatment" in X_final.columns)
print("Conversion column present:", "conversion" in X_final.columns)
print("Visit column present:", "visit" in X_final.columns)
print("Exposure column present:", "exposure" in X_final.columns)

Treatment column present: False
Conversion column present: False
Visit column present: False
Exposure column present: False


In [20]:
modeling_data = X_final.copy()

modeling_data["treatment"] = treatment.values
modeling_data["conversion"] = y_conversion.values
modeling_data["visit"] = y_visit.values
modeling_data["exposure"] = y_exposure.values

print("Modeling dataset shape:", modeling_data.shape)

modeling_data.head()

Modeling dataset shape: (100000, 39)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f0_f1_interaction,f2_f3_interaction,f4_f5_interaction,f6_f7_interaction,f8_f9_interaction,f10_f11_interaction,treatment,conversion,visit,exposure
0,-1.950935,-0.081747,2.453839,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,0.159482,0.718253,-0.034433,-0.249996,-0.015319,-0.022081,1,0,0,0
1,-1.950935,-0.081747,2.556092,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,0.159482,0.748183,-0.034433,-0.249996,-0.015319,-0.022081,1,0,0,0
2,-1.950935,-0.081747,2.408462,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,0.159482,0.704971,-0.034433,-0.249996,-0.015319,-0.022081,1,0,0,0
3,-1.950935,-0.081747,2.556526,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,0.159482,0.748310,-0.034433,-0.249996,-0.015319,-0.022081,1,0,0,0
4,-1.950935,-0.081747,2.693578,0.292706,-0.178693,0.192692,1.150615,-0.217272,0.058287,-0.262818,...,0.159482,0.788426,-0.034433,-0.249996,-0.015319,-0.022081,1,0,0,0


In [21]:
print("Conversion distribution:")
print(y_conversion.value_counts())

print("\nConversion rate:")
print(y_conversion.mean())

Conversion distribution:
conversion
0    99494
1      506
Name: count, dtype: int64

Conversion rate:
0.00506


In [22]:
print("Treatment distribution:")
print(treatment.value_counts())

print("\nTreatment proportions:")
print(treatment.value_counts(normalize=True))

Treatment distribution:
treatment
1    100000
Name: count, dtype: int64

Treatment proportions:
treatment
1    1.0
Name: proportion, dtype: float64


In [23]:
OUTPUT_DIR = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\data\processed"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = OUTPUT_DIR / "engineered_uplift_sample.csv"

modeling_data.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("File size (MB):", round(output_path.stat().st_size / (1024 ** 2), 2))

Saved: C:\Users\ugand\customer-churn-uplift-modeling\data\processed\engineered_uplift_sample.csv
File size (MB): 54.57


In [24]:
feature_report = pd.DataFrame({
    "metric": [
        "Original features",
        "Final engineered features",
        "Rows processed",
        "Treatment column included separately",
        "Conversion column included separately"
    ],
    "value": [
        len(feature_cols),
        X_final.shape[1],
        len(X_final),
        True,
        True
    ]
})

feature_report

,metric,value
0,Original features,12
1,Final engineered features,35
2,Rows processed,100000
3,Treatment column included separately,True
4,Conversion column included separately,True


In [25]:
print("Missing values:", X_final.isnull().sum().sum())
print("Infinite values:", np.isinf(X_final.select_dtypes(include=np.number)).sum().sum())

print("Treatment in X:", "treatment" in X_final.columns)
print("Conversion in X:", "conversion" in X_final.columns)

print("\nFinal X shape:", X_final.shape)
print("Validation complete.")

Missing values: 0
Infinite values: 0
Treatment in X: False
Conversion in X: False

Final X shape: (100000, 35)
Validation complete.


In [ ]:
# Conclusion

## Feature Engineering Summary

The Criteo Uplift dataset has been transformed into a machine-learning-ready feature representation.

### Key steps completed

- Selected the 12 anonymized customer features (`f0`–`f11`).
- Separated customer features from treatment assignment and outcomes.
- Validated feature data types and missing values.
- Checked for infinite numerical values.
- Standardized the numerical feature space.
- Created rank-based representations of the anonymized features.
- Created selected feature interaction terms.
- Generated aggregate customer-level feature statistics.
- Verified that treatment and outcome variables were not included in the base feature matrix.
- Created a modeling dataset containing engineered features, treatment assignment, and outcomes.
- Saved the engineered development dataset for downstream experimentation.

### Modeling Readiness

The resulting feature matrix is suitable for the next stage of the project.

The next stage will establish **baseline predictive models** and then progress toward **individual treatment-effect and uplift modeling**.

The treatment variable will remain explicitly separated from customer covariates so that the modeling pipeline can estimate differential treatment response rather than simply predict conversion.